# Session 3: solve the heat equation with a network, and recover the diffusivity
**A. Real-world problem.** A thin metal rod of length 1 (scaled units) has both ends held at
temperature 0 and starts with a smooth hump of heat, `u(x, 0) = sin(pi x)`. Heat leaks out and
the rod cools according to the heat equation `u_t = alpha u_xx`, where `alpha` is the
diffusivity. The exact answer is `sin(pi x) exp(-alpha pi^2 t)`. We ask a network `u(x, t)` to
satisfy the equation, the ends and the start, and we check it against the exact answer and
against a classical finite-difference solver. Then we turn the problem around: given a few
noisy readings, find `alpha`.

**Honest warning.** For this easy forward problem a classical solver is far cheaper and more
accurate. The point is the recipe, and the inverse problem where the network earns its keep.
This is a simplified teaching model, not a validated engineering tool.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
ALPHA = 0.1          # true diffusivity (scaled units: length^2 per time)

## Step 1: the network and its derivatives, by hand
One hidden layer of H tanh units: with `z_j = a_j x + b_j t + d_j` and `th_j = tanh(z_j)`,

    u = sum_j c_j th_j + e

Let `s = 1 - th^2` (the slope of tanh), `s2 = -2 th s` (its second derivative) and
`s3 = s (6 th^2 - 2)` (its third derivative). The chain rule gives

    u_t  = sum_j c_j b_j s_j
    u_xx = sum_j c_j a_j^2 s2_j

The residual of the heat equation is `r = u_t - alpha u_xx`.

In [ ]:
def init_net(n_units, seed=0):
    rng = np.random.default_rng(seed)
    return {"a": rng.normal(0.0, 2.0, n_units), "b": rng.normal(0.0, 1.0, n_units),
            "d": rng.normal(0.0, 1.0, n_units), "c": rng.normal(0.0, 1.0 / np.sqrt(n_units), n_units),
            "e": np.zeros(1)}


def forward(P, x, t):
    """Return u, u_t, u_xx at the points (x_i, t_i) and a cache for the backward pass."""
    th = np.tanh(np.outer(x, P["a"]) + np.outer(t, P["b"]) + P["d"])
    s = 1.0 - th ** 2
    s2 = -2.0 * th * s
    s3 = s * (6.0 * th ** 2 - 2.0)
    u = th @ P["c"] + P["e"][0]
    u_t = s @ (P["c"] * P["b"])
    u_xx = s2 @ (P["c"] * P["a"] ** 2)
    return u, u_t, u_xx, (th, s, s2, s3)

## Step 2: backpropagation through the derivatives
A loss depends on the network through `u`, `u_t` and `u_xx` at the points. Given how the loss
changes with each of them (`gu`, `gut`, `guxx`) the parameter gradients are sums of

    u:     dc = th        da = c s x                       db = c s t                         dd = c s
    u_t:   dc = b s       da = c b s2 x                    db = c (s + b s2 t)                dd = c b s2
    u_xx:  dc = a^2 s2    da = c (2 a s2 + a^2 s3 x)       db = c a^2 s3 t                    dd = c a^2 s3

(each row is d(quantity)/d(parameter) for hidden unit j; "x" and "t" are the point coordinates).

In [ ]:
def backward(P, x, t, cache, gu, gut, guxx):
    th, s, s2, s3 = cache
    a, b, c = P["a"], P["b"], P["c"]
    X, T = x[:, None], t[:, None]
    gu_, gut_, guxx_ = gu[:, None], gut[:, None], guxx[:, None]
    return {
        "c": gu @ th + gut @ (s * b) + guxx @ (s2 * a ** 2),
        "a": (gu_ * c * s * X + gut_ * c * b * s2 * X + guxx_ * c * (2 * a * s2 + a ** 2 * s3 * X)).sum(axis=0),
        "b": (gu_ * c * s * T + gut_ * c * (s + b * s2 * T) + guxx_ * c * a ** 2 * s3 * T).sum(axis=0),
        "d": (gu_ * c * s + gut_ * c * b * s2 + guxx_ * c * a ** 2 * s3).sum(axis=0),
        "e": np.array([gu.sum()]),
    }

## Step 3: the key correctness check
Compare the formulas for `u_t` and `u_xx` with central finite differences of `u` itself, and
the parameter gradients with finite differences of a loss. If either disagrees, training
would silently solve the wrong problem.

In [ ]:
def derivative_check(h=1e-4):
    """Largest gaps: (u_t formula vs finite difference, u_xx formula vs finite difference)."""
    P = init_net(8, 0)
    x = np.array([0.3, 0.7, 0.45])
    t = np.array([0.2, 0.5, 0.9])
    u, u_t, u_xx, _ = forward(P, x, t)
    fd_t = (forward(P, x, t + h)[0] - forward(P, x, t - h)[0]) / (2 * h)
    fd_xx = (forward(P, x + h, t)[0] - 2 * u + forward(P, x - h, t)[0]) / h ** 2
    return float(np.abs(u_t - fd_t).max()), float(np.abs(u_xx - fd_xx).max())


def gradient_check(h=1e-6):
    """Largest relative gap between hand gradients and finite-difference gradients of a test loss."""
    P = init_net(6, 1)
    x = np.array([0.3, 0.7])
    t = np.array([0.2, 0.5])

    def loss(Q):
        u, u_t, u_xx, _ = forward(Q, x, t)
        return float(np.sum(u ** 2) + 0.7 * np.sum(u_t ** 2) + 0.3 * np.sum(u_xx ** 2))

    u, u_t, u_xx, cache = forward(P, x, t)
    G = backward(P, x, t, cache, 2 * u, 2 * 0.7 * u_t, 2 * 0.3 * u_xx)
    worst = 0.0
    for key in P:
        for i in range(len(P[key])):
            Pp = {k: v.copy() for k, v in P.items()}
            Pm = {k: v.copy() for k, v in P.items()}
            Pp[key][i] += h
            Pm[key][i] -= h
            fd = (loss(Pp) - loss(Pm)) / (2 * h)
            worst = max(worst, abs(fd - G[key][i]) / max(1.0, abs(fd)))
    return worst

## Step 4: the loss and the training loop (Adam, by hand)
Three pieces, each a mean of squares:

    residual  = mean of (u_t - alpha u_xx)^2 at N random points inside the rod and time span
    boundary  = mean of u(0, t)^2 and u(1, t)^2 at points along the two ends
    initial   = mean of (u(x, 0) - sin(k pi x))^2 along the start

    loss = w_res * residual + boundary + initial

`w_res` is the weight of the physics term. For the inverse problem a fourth piece compares
the network with the noisy readings and `alpha` becomes a trainable number.

In [ ]:
def exact(x, t, alpha=ALPHA, k=1):
    return np.sin(k * np.pi * x) * np.exp(-alpha * (k * np.pi) ** 2 * t)


def _adam(P, G, state, lr, step, lr_scale=None):
    m, v = state
    for key in P:
        m[key] = 0.9 * m[key] + 0.1 * G[key]
        v[key] = 0.999 * v[key] + 0.001 * G[key] ** 2
        rate = lr * (lr_scale or {}).get(key, 1.0)
        P[key] -= rate * (m[key] / (1 - 0.9 ** step)) / (np.sqrt(v[key] / (1 - 0.999 ** step)) + 1e-8)


def train_pinn(alpha=ALPHA, n_units=20, steps=1000, n_col=200, w_res=1.0, k=1, seed=0, lr=5e-3,
               readings=None, alpha_start=None, w_data=1.0, record_every=50):
    """Train a PINN. With `readings=(x, t, u)` and `alpha_start`, alpha is a trainable parameter.

    Returns (P, alpha_final, history) where history has the loss pieces and alpha every `record_every` steps."""
    rng = np.random.default_rng(seed)
    P = init_net(n_units, seed)
    xc, tc = rng.random(n_col), rng.random(n_col)
    s_ = np.linspace(0.0, 1.0, 50)
    xb, tb = np.r_[np.zeros(50), np.ones(50)], np.r_[s_, s_]
    xi, ti = s_, np.zeros(50)
    ui_target = np.sin(k * np.pi * xi)
    inverse = readings is not None
    if inverse:
        P["alpha"] = np.array([float(alpha_start)])
        xo, to, uo = readings
    state = ({key: np.zeros_like(v) for key, v in P.items()}, {key: np.zeros_like(v) for key, v in P.items()})
    hist = {"step": [], "residual": [], "boundary": [], "initial": [], "data": [], "alpha": []}
    for step in range(1, steps + 1):
        al = P["alpha"][0] if inverse else alpha
        u, u_t, u_xx, cache = forward(P, xc, tc)
        r = u_t - al * u_xx
        ub, _, _, cache_b = forward(P, xb, tb)
        ui, _, _, cache_i = forward(P, xi, ti)
        g_r = w_res * 2.0 * r / n_col
        G1 = backward(P, xc, tc, cache, np.zeros_like(r), g_r, -al * g_r)
        G2 = backward(P, xb, tb, cache_b, 2.0 * ub / len(xb), np.zeros_like(ub), np.zeros_like(ub))
        G3 = backward(P, xi, ti, cache_i, 2.0 * (ui - ui_target) / len(xi), np.zeros_like(ui), np.zeros_like(ui))
        G = {key: G1[key] + G2[key] + G3[key] for key in G1}
        data_loss = 0.0
        if inverse:
            uu, _, _, cache_o = forward(P, xo, to)
            G4 = backward(P, xo, to, cache_o, w_data * 2.0 * (uu - uo) / len(xo), np.zeros_like(uu), np.zeros_like(uu))
            G = {key: G[key] + G4[key] for key in G4}
            G["alpha"] = np.array([-np.sum(g_r * u_xx)])
            data_loss = float(np.mean((uu - uo) ** 2))
        _adam(P, G, state, lr, step, {"alpha": 2.0} if inverse else None)
        if step % record_every == 0 or step == 1:
            hist["step"].append(step)
            hist["residual"].append(float(np.mean(r ** 2)))
            hist["boundary"].append(float(np.mean(ub ** 2)))
            hist["initial"].append(float(np.mean((ui - ui_target) ** 2)))
            hist["data"].append(data_loss)
            hist["alpha"].append(float(al))
    alpha_final = float(P["alpha"][0]) if inverse else float(alpha)
    return P, alpha_final, hist


def rel_l2_error(P, alpha=ALPHA, k=1, n=101):
    """Relative L2 error of the network against the exact solution on an n x n grid of (x, t)."""
    g = np.linspace(0.0, 1.0, n)
    X, T = np.meshgrid(g, g)
    x, t = X.ravel(), T.ravel()
    u = forward(P, x, t)[0]
    ex = exact(x, t, alpha, k)
    return float(np.linalg.norm(u - ex) / np.linalg.norm(ex))

## Step 5: the classical reference
Explicit finite differences on `nx` intervals: each interior value moves towards the average
of its two neighbours,

    u_i(new) = u_i + r (u_{i-1} - 2 u_i + u_{i+1}),    r = alpha dt / dx^2

This is stable only for `r <= 1/2`. With a tiny alternating ripple added to the start
(`noise`), an unstable `r` makes the ripple explode.

In [ ]:
def fd_heat(alpha=ALPHA, nx=20, r=0.4, t_end=1.0, noise=0.0):
    """Explicit scheme to time t_end. Returns (x, u, n_steps, dt)."""
    dx = 1.0 / nx
    x = np.linspace(0.0, 1.0, nx + 1)
    dt = r * dx ** 2 / alpha
    n_steps = int(round(t_end / dt))
    u = np.sin(np.pi * x)
    u[1:-1] += noise * (-1.0) ** np.arange(1, nx)
    for _ in range(n_steps):
        u[1:-1] = u[1:-1] + r * (u[:-2] - 2 * u[1:-1] + u[2:])
        u[0] = u[-1] = 0.0
    return x, u, n_steps, dt


def fd_error(alpha=ALPHA, nx=20, r=0.4, t_end=1.0):
    x, u, n_steps, dt = fd_heat(alpha, nx, r, t_end)
    ex = exact(x, n_steps * dt)
    return float(np.linalg.norm(u - ex) / np.linalg.norm(ex))


def pinn_error_at_time(P, t_end=1.0, alpha=ALPHA, n=101):
    x = np.linspace(0.0, 1.0, n)
    t = np.full(n, t_end)
    ex = exact(x, t, alpha)
    return float(np.linalg.norm(forward(P, x, t)[0] - ex) / np.linalg.norm(ex))

## Step 6: the inverse problem
Thirty noisy readings of the temperature at random places and times, `alpha` unknown. We keep
the same network and loss, add a data term, and let `alpha` be trained together with the
weights. The classical alternative when the form of the answer is known is to fit one number
to the readings directly; we compute that too.

In [ ]:
def make_readings(alpha=ALPHA, n=30, sigma=0.02, seed=0):
    rng = np.random.default_rng(seed + 1000)
    x, t = rng.uniform(0.05, 0.95, n), rng.uniform(0.05, 1.0, n)
    return x, t, exact(x, t, alpha) + sigma * rng.normal(size=n)


def fit_alpha_direct(readings):
    """Best alpha for u = sin(pi x) exp(-alpha pi^2 t) by scanning (valid only if that form is known)."""
    x, t, u = readings
    grid = np.linspace(0.005, 0.6, 5951)
    sse = ((np.sin(np.pi * x)[None, :] * np.exp(-grid[:, None] * np.pi ** 2 * t[None, :]) - u[None, :]) ** 2).sum(axis=1)
    return float(grid[np.argmin(sse)])

In [ ]:
if __name__ == "__main__":
    import time

    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    d_t, d_xx = derivative_check()
    print(f"u_t formula vs finite difference: {d_t:.1e};  u_xx formula vs finite difference: {d_xx:.1e}")
    print(f"parameter gradients vs finite differences, worst relative gap: {gradient_check():.1e}")

    t0 = time.time()
    P, _, hist = train_pinn()
    err = rel_l2_error(P)
    print(f"PINN (20 units, 200 points, 1000 steps): relative L2 error {err:.4f}  ({time.time() - t0:.1f} s)")
    print(f"final loss pieces: residual {hist['residual'][-1]:.2e}, boundary {hist['boundary'][-1]:.2e}, "
          f"initial {hist['initial'][-1]:.2e}")
    x_fd, u_fd, n_steps, dt = fd_heat()
    print(f"finite differences (20 intervals, r = 0.4): {n_steps} steps of dt = {dt:.2f}, error at t = 1: "
          f"{fd_error():.5f}; PINN error at t = 1: {pinn_error_at_time(P):.4f}")
    x_u, u_u, _, _ = fd_heat(r=0.6, noise=1e-3)
    print(f"unstable r = 0.6: largest |u| at t = 1 is {np.abs(u_u).max():.1e} (should be below 1)")

    print("\nweight of the physics term (1000 steps):")
    for w in (0.01, 1.0, 100.0):
        Pw = P if w == 1.0 else train_pinn(w_res=w)[0]
        print(f"  w_res = {w:6.2f}: relative L2 error {rel_l2_error(Pw):.4f}")
    print("number of collocation points:")
    for n in (5, 20, 200):
        Pn = P if n == 200 else train_pinn(n_col=n)[0]
        print(f"  N = {n:3d}: relative L2 error {rel_l2_error(Pn):.4f}")
    print("higher spatial frequency k (initial hump sin(k pi x)), same network:")
    for k in (1, 3):
        Pk = P if k == 1 else train_pinn(k=k, steps=1500)[0]
        print(f"  k = {k}: relative L2 error {rel_l2_error(Pk, k=k):.4f}")

    readings = make_readings()
    t0 = time.time()
    Pi, a_fin, hist_i = train_pinn(readings=readings, alpha_start=0.3, steps=2000)
    print(f"\ninverse: alpha started at 0.3, recovered {a_fin:.4f} (true {ALPHA}); direct fit {fit_alpha_direct(readings):.4f} "
          f"({time.time() - t0:.1f} s)")

    # ---- figures
    g = np.linspace(0, 1, 101)
    X, T = np.meshgrid(g, g)
    U = forward(P, X.ravel(), T.ravel())[0].reshape(X.shape)
    E = exact(X, T)
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
    for ax, Z, ttl, vmax in zip(axs, (E, U, np.abs(U - E)), ("exact", "network", "absolute error"), (1, 1, None)):
        im = ax.imshow(Z, origin="lower", extent=[0, 1, 0, 1], aspect="auto", cmap="inferno" if vmax else "viridis", vmin=0 if vmax else None, vmax=vmax)
        ax.set_title(ttl)
        ax.set_xlabel("position x (scaled units)")
        fig.colorbar(im, ax=ax, label="temperature u" if vmax else "|u - exact|")
    axs[0].set_ylabel("time t (scaled units)")
    fig.tight_layout()
    fig.savefig(OUT / "heatmaps.png", dpi=110)

    fig2, axs2 = plt.subplots(1, 2, figsize=(10, 4))
    for key in ("residual", "boundary", "initial"):
        axs2[0].semilogy(hist["step"], hist[key], label=key)
    axs2[0].set_xlabel("training step")
    axs2[0].set_ylabel("mean squared loss piece (scaled units^2)")
    axs2[0].set_title("The three loss pieces during training")
    axs2[0].legend()
    axs2[0].grid(alpha=0.3, which="both")
    for tt, col in ((0.0, "tab:blue"), (0.5, "tab:green"), (1.0, "tab:red")):
        axs2[1].plot(g, exact(g, tt), "--", color=col)
        axs2[1].plot(g, forward(P, g, np.full_like(g, tt))[0], color=col, label=f"t = {tt}")
    axs2[1].plot(x_fd, u_fd, "k.", label="finite differences, t = 1")
    axs2[1].set_xlabel("position x (scaled units)")
    axs2[1].set_ylabel("temperature u (scaled units)")
    axs2[1].set_title("Profiles: network (solid) vs exact (dashed)")
    axs2[1].legend(fontsize=8)
    axs2[1].grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "loss_and_profiles.png", dpi=110)

    fig3, ax3 = plt.subplots(figsize=(6.5, 4))
    ax3.plot(hist_i["step"], hist_i["alpha"], label="alpha during training")
    ax3.axhline(ALPHA, color="k", ls="--", label="true alpha = 0.1")
    ax3.set_xlabel("training step")
    ax3.set_ylabel("diffusivity alpha (scaled units)")
    ax3.set_title("Inverse problem: alpha recovered from 30 noisy readings")
    ax3.grid(alpha=0.3)
    ax3.legend()
    fig3.tight_layout()
    fig3.savefig(OUT / "inverse_alpha.png", dpi=120)

## Where this stops
One space dimension, a smooth start, a one-hidden-layer network and a few thousand steps.
Sharper or faster-oscillating solutions, several dimensions and long times are much harder for
this recipe. That is the road to the Target: see `promo/roadmap.md`.

## Coding tasks
1. **Easy.** Change `w_res` to 10 in `train_pinn` and print the relative L2 error; place it between the values 1 and 100 from the table before running.
2. **Stretch.** Write `fd_error_for_r(r)` that returns the finite-difference error at t = 1 for r = 0.1, 0.25 and 0.4 and explain the trend.
3. **Explain.** Explain why the inverse problem still needs the boundary and initial pieces in the loss (try removing the initial piece and see what happens to alpha).

## Assessment
- **A. Conceptual.** Why can both a too small and a too large weight of the residual term w_res make the network worse?
- **B. Numerical.** The diffusivity is doubled to 0.2. Find the centre temperature at t = 1 and compare with alpha = 0.1. For dx = 0.05 and dt = 0.01, what is r now, and is the explicit scheme stable?
- **C. Graph and simulation interpretation.** In the loss figure, which piece starts the largest and how far do the pieces fall? What does the profile panel show about the time t = 1?
- **D. Design and optimisation challenge.** Design: you want a relative L2 error below 0.03 at the smallest training cost by changing the number of collocation points. Which of N = 5, 20 and 200 would you choose, and what is the risk of N = 5?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    d_t, d_xx = L.derivative_check()
    checks.append((f"u_t formula matches a finite difference (gap {d_t:.1e})", d_t < 1e-6))
    checks.append((f"u_xx formula matches a finite difference (gap {d_xx:.1e})", d_xx < 1e-5))
    gap = L.gradient_check()
    checks.append((f"hand-coded parameter gradients match finite differences (gap {gap:.1e})", gap < 1e-6))

    # exact solution: hand calculations
    checks.append(("worked example: exact u(0.5, 1) = exp(-0.1 pi^2) = 0.3727", abs(float(L.exact(0.5, 1.0)) - 0.3727) < 5e-5))
    t_half = np.log(2) / (L.ALPHA * np.pi ** 2)
    checks.append(("half-life ln 2 / (alpha pi^2) = 0.702 and the exact amplitude is 0.5 there",
                   abs(t_half - 0.702) < 5e-4 and abs(float(L.exact(0.5, t_half)) - 0.5) < 1e-12))
    h = 1e-4
    x0, t0 = 0.37, 0.4
    ut = (L.exact(x0, t0 + h) - L.exact(x0, t0 - h)) / (2 * h)
    uxx = (L.exact(x0 + h, t0) - 2 * L.exact(x0, t0) + L.exact(x0 - h, t0)) / h ** 2
    checks.append(("the exact solution has zero residual u_t - alpha u_xx", abs(ut - L.ALPHA * uxx) < 1e-5))
    checks.append(("edge: exact solution is 0 at both ends and sin(pi x) at t = 0",
                   abs(float(L.exact(0.0, 0.7))) < 1e-15 and abs(float(L.exact(1.0, 0.7))) < 1e-12 and abs(float(L.exact(0.25, 0.0)) - np.sin(np.pi / 4)) < 1e-15))

    checks.append(("hand calculation: with alpha = 0.2 the amplitude at t = 1 is exp(-0.2 pi^2) = 0.1389; r = 0.1 * 0.01 / 0.05^2 = 0.4",
                   abs(float(L.exact(0.5, 1.0, 0.2)) - 0.1389) < 5e-5 and abs(L.ALPHA * 0.01 / 0.05 ** 2 - 0.4) < 1e-12))

    # main run
    P, _, hist = L.train_pinn()
    err = L.rel_l2_error(P)
    checks.append((f"worked example: PINN relative L2 error is below 0.02 (got {err:.4f})", err < 0.02))
    u_net = float(L.forward(P, np.array([0.5]), np.array([1.0]))[0][0])
    checks.append((f"network value at (0.5, 1) is within 0.01 of 0.3727 (got {u_net:.4f})", abs(u_net - 0.3727) < 0.01))
    checks.append(("all three loss pieces end below 1e-3 and the residual fell by 50x and boundary and initial by 1000x",
                   max(hist["residual"][-1], hist["boundary"][-1], hist["initial"][-1]) < 1e-3 and hist["residual"][0] > 50 * hist["residual"][-1] and hist["boundary"][0] > 1000 * hist["boundary"][-1] and hist["initial"][0] > 1000 * hist["initial"][-1]))

    # classical reference
    e_fd = L.fd_error()
    e_pinn_t1 = L.pinn_error_at_time(P)
    checks.append((f"finite differences (20 intervals, r = 0.4) error at t = 1 is below 0.005 (got {e_fd:.5f})", e_fd < 0.005))
    checks.append((f"the PINN is less accurate than finite differences at t = 1 ({e_pinn_t1:.3f} vs {e_fd:.5f})", e_pinn_t1 > 5 * e_fd))
    _, _, n_steps, dt = L.fd_heat()
    fd_updates = n_steps * 19
    pinn_evals = 1000 * (200 + 100 + 50) * 20
    checks.append((f"work: finite differences 100 steps x 19 nodes = {fd_updates} updates; PINN {pinn_evals} unit evaluations (over 1000 times more)",
                   n_steps == 100 and abs(dt - 0.01) < 1e-12 and pinn_evals > 1000 * fd_updates))
    _, u_stable, _, _ = L.fd_heat(noise=1e-3)
    _, u_bad, _, _ = L.fd_heat(r=0.6, noise=1e-3)
    checks.append((f"stability: r = 0.4 stays below 1 while r = 0.6 reaches {np.abs(u_bad).max():.1e}", np.abs(u_stable).max() < 1 and np.abs(u_bad).max() > 1e3))

    # sensitivity and failure cases
    e = {}
    for key, kw in (("w001", dict(w_res=0.01)), ("w100", dict(w_res=100.0)), ("n5", dict(n_col=5)), ("n20", dict(n_col=20))):
        e[key] = L.rel_l2_error(L.train_pinn(**kw)[0])
    checks.append((f"weight of physics term: 0.01 gives {e['w001']:.3f}, 1 gives {err:.3f}, 100 gives {e['w100']:.3f}; both extremes are worse",
                   e["w001"] > 0.1 and e["w100"] > 0.05 and err < e["w001"] and err < e["w100"]))
    checks.append((f"too few collocation points: N = 5 gives {e['n5']:.3f} (above 0.15); N = 20 gives {e['n20']:.3f} (below 0.03)", e["n5"] > 0.15 and e["n20"] < 0.03))
    Pk, _, _ = L.train_pinn(k=3, steps=1500)
    e_k3 = L.rel_l2_error(Pk, k=3)
    checks.append((f"spectral bias: a start of sin(3 pi x) gives error {e_k3:.2f}, above 0.3 (sin(pi x) gave {err:.3f})", e_k3 > 0.3))

    # inverse problem
    readings = L.make_readings()
    Pi, a_fin, hi = L.train_pinn(readings=readings, alpha_start=0.3, steps=2000)
    checks.append((f"inverse: alpha recovered {a_fin:.3f} from a start of 0.3, within 5 percent of 0.1", abs(a_fin - 0.1) < 0.005 and hi["alpha"][0] == 0.3))
    a_dir = L.fit_alpha_direct(readings)
    checks.append((f"classical direct fit of alpha is {a_dir:.3f}, also within 5 percent (noise limits both)", abs(a_dir - 0.1) < 0.005))
    checks.append(("roadmap: the shipped inverse run misses the truth by more than 2 percent", abs(a_fin - 0.1) / 0.1 > 0.02))

    # mini-project: true alpha 0.15, start 0.5, 30 readings with noise 0.02
    R = L.make_readings(0.15, 30, 0.02, 0)
    Pm, a_m, hm = L.train_pinn(alpha=0.0, readings=R, alpha_start=0.5, steps=2000)
    rel = np.abs(np.array(hm["alpha"]) - 0.15) / 0.15
    steps = np.array(hm["step"])
    first_pass = int(steps[np.argmax(rel < 0.1)])
    settle = int(steps[max(i for i in range(len(rel)) if rel[i] >= 0.1) + 1])
    checks.append((f"mini-project: alpha is first within 10 percent at step {first_pass} (a false alarm) but settles only from step {settle}",
                   first_pass == 200 and settle == 1200))
    checks.append((f"mini-project: final alpha {a_m:.3f} is within 10 percent of 0.15; direct fit {L.fit_alpha_direct(R):.3f}",
                   abs(a_m - 0.15) / 0.15 < 0.1 and abs(L.fit_alpha_direct(R) - 0.15) / 0.15 < 0.05))

    # roadmap: shipped boundary values are not exactly zero
    g = np.linspace(0, 1, 51)
    edge = max(np.abs(L.forward(P, np.zeros_like(g), g)[0]).max(), np.abs(L.forward(P, np.ones_like(g), g)[0]).max())
    checks.append((f"roadmap: the shipped network only approximates the end temperatures (largest |u| at an end {edge:.3f}, above 0.01)", edge > 0.01))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
